In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Tuple

import numpy as np
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler


@dataclass
class IrisDataset:
    features: np.ndarray
    labels: np.ndarray
    target_names: np.ndarray

In [2]:
def load_iris_data() -> IrisDataset:
    """Load the Iris benchmark dataset and return a structured container."""
    raw = load_iris()
    return IrisDataset(features=raw.data, labels=raw.target, target_names=raw.target_names)


In [3]:
def prepare_data(
    dataset: IrisDataset,
    test_size: float = 0.25,
    random_state: int = 42,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, StandardScaler]:
    """Split the data, scale the features, and return training/test sets."""
    x_train, x_test, y_train, y_test = train_test_split(
        dataset.features,
        dataset.labels,
        test_size=test_size,
        stratify=dataset.labels,
        random_state=random_state,
    )

    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)

    return x_train_scaled, x_test_scaled, y_train, y_test, scaler


In [4]:
def build_knn_classifier(n_neighbors: int = 5) -> KNeighborsClassifier:
    """Create a KNN classifier with a sensible default K value."""
    return KNeighborsClassifier(n_neighbors=n_neighbors)



In [5]:
def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray, target_names: np.ndarray) -> None:
    """Print classification metrics and a confusion matrix in a human-readable format."""
    print("Model evaluation")
    print("==============")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}\n")
    print("Classification report:")
    print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

    cm = confusion_matrix(y_true, y_pred)
    print("Confusion matrix:")
    print(cm)
    print()

    print("Class-level TP / FP / FN / TN")
    print("--------------------------------")
    for index, class_name in enumerate(target_names):
        tp = cm[index, index]
        fp = cm[:, index].sum() - tp
        fn = cm[index, :].sum() - tp
        tn = cm.sum() - tp - fp - fn
        print(
            f"{class_name:10s}: TP={tp:2d}  FP={fp:2d}  FN={fn:2d}  TN={tn:2d}"
        )


In [6]:
def main() -> None:
    dataset = load_iris_data()
    x_train, x_test, y_train, y_test, scaler = prepare_data(dataset)

    print("Project 2: Data Classification Using AI")
    print("Iris dataset loaded with 150 samples and 4 numerical features.")
    print(f"Training samples: {x_train.shape[0]}, Test samples: {x_test.shape[0]}\n")

    model = build_knn_classifier(n_neighbors=5)
    model.fit(x_train, y_train)
    predictions = model.predict(x_test)

    evaluate_predictions(y_test, predictions, dataset.target_names)

    print("The pipeline demonstrates how supervised learning replaces static rules")
    print("with a model that generalizes from training examples to new observations.")


if __name__ == "__main__":
    main()

Project 2: Data Classification Using AI
Iris dataset loaded with 150 samples and 4 numerical features.
Training samples: 112, Test samples: 38

Model evaluation
Accuracy: 0.9211

Classification report:
              precision    recall  f1-score   support

      setosa     1.0000    1.0000    1.0000        12
  versicolor     0.8125    1.0000    0.8966        13
   virginica     1.0000    0.7692    0.8696        13

    accuracy                         0.9211        38
   macro avg     0.9375    0.9231    0.9220        38
weighted avg     0.9359    0.9211    0.9200        38

Confusion matrix:
[[12  0  0]
 [ 0 13  0]
 [ 0  3 10]]

Class-level TP / FP / FN / TN
--------------------------------
setosa    : TP=12  FP= 0  FN= 0  TN=26
versicolor: TP=13  FP= 3  FN= 0  TN=22
virginica : TP=10  FP= 0  FN= 3  TN=25
The pipeline demonstrates how supervised learning replaces static rules
with a model that generalizes from training examples to new observations.
